In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('cleaned_data')

# apply exclusion criteria to remove outliers
df = df[~(
    (df['sbp'] < 90) | (df['sbp'] > 200) |
    (df['tc'] < 130) | (df['tc'] > 320) |
    (df['hdlc'] < 20) | (df['hdlc'] > 100) |
    (df['bmi'] < 18.5) | (df['bmi'] >= 40))]

df['deprivation_bin'] = pd.cut(
    df['deprivation_index'],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=[1, 2, 3, 4],
    include_lowest=True
).astype(int)
df['race'] = df['race'].replace('Black or African American', 'Black')

df.head()

## eGFR Calculation

In [ ]:
scre = df['scre'].to_numpy()
age = df['age'].to_numpy()
df['sex'] = (df['sex'] == 'Male').astype(int)
sex = df['sex'].to_numpy()

egfr_female = 141.57 * np.minimum(scre / 0.7, 1) ** -0.241 * np.maximum(scre / 0.7, 1) ** -1.2 * 0.993839 ** age
egfr_male = 141.57 * np.minimum(scre / 0.9, 1) ** -0.302 * np.maximum(scre / 0.9, 1) ** -1.2 * 0.993839 ** age
df['egfr'] = np.where(sex == 1, egfr_male, egfr_female)

## Statin Indicator by ACC/AHA 2013 Guidelines

In [ ]:
import pandas as pd

def assign_statin_use(df, gender_column='sex'):
    """
    Assigns a binary 'statin' variable based on cholesterol, HDL, and SBP.
    Assumes df has columns: 'total_cholesterol', 'hdl_cholesterol', 'sbp', and optionally 'gender'.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        gender_column (str): Name of the column indicating gender ('male' or 'female').

    Returns:
        pd.Series: A binary Series for statin use (1 = likely on statin, 0 = not likely).
    """

    # HDL threshold based on gender
    def hdl_threshold(row):
        return 40 if row[gender_column] == 1 else 50

    hdl_thresh = df.apply(hdl_threshold, axis=1)
    
    # Calculate total cholesterol / HDL ratio
    chol_hdl_ratio = df['tc'] / df['hdlc']

    # Apply the rule
    statin = (
        (df['tc'] >= 240) |
        (df['hdlc'] <= hdl_thresh) |
        (chol_hdl_ratio > 5) |
        ((df['sbp'] > 140) & (df['tc'] >= 200))
    ).astype(int)

    return statin

df['statin'] = assign_statin_use(df)

## PREVENT

### Data Transformation

In [ ]:
# PREVENT Variables
df['age'] = (df['age'] - 55) / 10
df['nonhdlc'] = (df['tc'] - df['hdlc']) * 0.02586 - 3.5
df['hdlc'] = (df['hdlc'] * 0.02586 - 1.3) / 0.3
df['sbp1'] = (np.minimum(df['sbp'], 110) - 110) / 20
df['sbp2'] = (np.maximum(df['sbp'], 110) - 130) / 20
df['bmi1'] = (np.minimum(df['bmi'], 30) - 25) / 5
df['bmi2'] = (np.maximum(df['bmi'], 30) - 30) / 5
df['egfr1'] = (np.minimum(df['egfr'], 60) - 60) / -15
df['egfr2'] = (np.maximum(df['egfr'], 60) - 90) / -15
df['sbp2treat'] = df['sbp2'] * df['antihtn']
df['nonhdlctreat'] = df['nonhdlc'] * df['statin']
df['agenonhdlc'] = df['age'] * df['nonhdlc']
df['agehdlc'] = df['age'] * df['hdlc']
df['agesbp2'] = df['age'] * df['sbp2']
df['agediabetes'] = df['age'] * df['diabetes']
df['agecursmk'] = df['age'] * df['smoker']
df['agebmi2'] = df['age'] * df['bmi2']
df['ageegfr1'] = df['age'] * df['egfr1']
df['cons'] = 1

### Risk Calculation

In [ ]:
# Coefficient matrix for males
coefficient_male_cvd = np.array([0.7688528, 0.0736174, -0.0954431, -0.4347345, 0.3362658, 0.7692857,
                                 0.4386871, 0.5378979, 0.0164827, 0.288879, -0.1337349, -0.0475924,
                                 0.150273, -0.0517874, 0.0191169, -0.1049477, -0.2251948, -0.0895067,
                                 -0.1543702, -3.031168]).reshape(20, 1)

# Coefficient matrix for females
coefficient_female_cvd = np.array([ 0.7939329, 0.0305239, -0.1606857, -0.2394003, 0.360078, 0.8667604,
                                   0.5360739, 0.6045917, 0.0433769, 0.3151672, -0.1477655, -0.0663612,
                                   0.1197879,  -0.0819715, 0.0306769, -0.0946348, -0.27057, -0.078715,
                                   -0.1637806, -3.307728]).reshape(20, 1)
# Define covariate columns (order must match coefficient order)
covariate_cols = [
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'smoker',
    'egfr1', 'egfr2', 'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat',
    'agenonhdlc', 'agehdlc', 'agesbp2', 'agediabetes', 'agecursmk',
    'ageegfr1', 'cons']

# Compute risk for males
df_male = df[df['sex'] == 1].copy()
X_male = df_male[covariate_cols].values
lo_male = X_male @ coefficient_male_cvd
df_male['prevent_cvd'] = (np.exp(lo_male) / (1 + np.exp(lo_male))).flatten()

# Compute risk for females
df_female = df[df['sex'] == 0].copy()
X_female = df_female[covariate_cols].values
lo_female = X_female @ coefficient_female_cvd
df_female['prevent_cvd'] = (np.exp(lo_female) / (1 + np.exp(lo_female))).flatten()

# Combine and sort to match original index order
df_updated = pd.concat([df_male, df_female]).sort_index()

# Assign back to original df
df.loc[df_male.index, 'prevent_cvd'] = df_male['prevent_cvd']
df.loc[df_female.index, 'prevent_cvd'] = df_female['prevent_cvd']

## Local ML Model

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

# Define predictor columns
predictors = [
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'smoker',
    'egfr1', 'egfr2', 'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat',
    'agenonhdlc', 'agehdlc', 'agesbp2', 'agediabetes', 'agecursmk',
    'ageegfr1'
]

# Target variable
target = 'cvd'

# Initialize prediction column
df['xgb_cvd'] = np.nan

# Function for stratified CV and assignment
def run_stratified_xgb_cv(df_subgroup, original_df, n_splits=5, seed=705):
    subset = df_subgroup[predictors + [target]]
    X = subset[predictors]
    y = subset[target]
    indices = subset.index

    oof_preds = np.zeros(len(subset))

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        model = XGBClassifier(
            n_estimators=500,         # more trees with early stopping
            max_depth=3,              # shallower trees generalize better
            learning_rate=0.03,       # slower learning, more robust
            subsample=0.8,            # bagging: row sampling
            colsample_bytree=0.8,     # feature sampling per tree
            scale_pos_weight=9,       # handle class imbalance (90/10)
            use_label_encoder=False,  # keep default behavior
            eval_metric='logloss',    # or 'auc' if using early stopping
            random_state=seed)
        model.fit(X_train, y_train)
        oof_preds[test_idx] = model.predict_proba(X.iloc[test_idx])[:, 1]

    # Assign predictions back using original indices
    original_df.loc[indices, 'xgb_cvd'] = oof_preds

# Run on males
df_male = df[df['sex'] == 1]
run_stratified_xgb_cv(df_male, df)

# Run on females
df_female = df[df['sex'] == 0]
run_stratified_xgb_cv(df_female, df)

## Local NN Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Predictor and target columns
predictors = [
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'smoker',
    'egfr1', 'egfr2', 'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat',
    'agenonhdlc', 'agehdlc', 'agesbp2', 'agediabetes', 'agecursmk',
    'ageegfr1'
]
target = 'cvd'

# Init prediction column
df['nn_cvd'] = np.nan

# Neural network model
class CVDNet(nn.Module):
    def __init__(self, input_dim):
        super(CVDNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Cross-validation and training
def run_stratified_nn_cv(df_subgroup, original_df, n_splits=5, seed=705, epochs=30, batch_size=64):
    subset = df_subgroup[predictors + [target]]
    X = subset[predictors].values.astype(np.float32)
    y = subset[target].values.astype(np.float32)
    indices = subset.index

    # Normalize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    oof_preds = np.zeros(len(subset))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, test_idx in skf.split(X, y):
        X_train = torch.tensor(X[train_idx]).to(device)
        y_train = torch.tensor(y[train_idx]).unsqueeze(1).to(device)
        X_test = torch.tensor(X[test_idx]).to(device)

        # Model and optimizer
        model = CVDNet(input_dim=X.shape[1]).to(device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Training loop
        model.train()
        for epoch in range(epochs):
            perm = torch.randperm(X_train.size(0))
            for start in range(0, X_train.size(0), batch_size):
                end = start + batch_size
                xb = X_train[perm[start:end]]
                yb = y_train[perm[start:end]]

                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()

        # Inference
        model.eval()
        with torch.no_grad():
            preds = model(X_test).squeeze().cpu().numpy()
            oof_preds[test_idx] = preds

    original_df.loc[indices, 'nn_cvd'] = oof_preds

# Run on male and female subgroups
df_male = df[df['sex'] == 1]
run_stratified_nn_cv(df_male, df)

df_female = df[df['sex'] == 0]
run_stratified_nn_cv(df_female, df)

## Predictive Performance

In [ ]:
from sklearn.metrics import roc_auc_score

# Define subgroup conditions
subgroup_conditions = {
    "Male": (df['sex'] == 1),
    "Female": (df['sex'] == 0),
    "White Male": (df['race'] == 'White') & (df['sex'] == 1),
    "White Female": (df['race'] == 'White') & (df['sex'] == 0),
    "Black Male": (df['race'] == 'Black') & (df['sex'] == 1),
    "Black Female": (df['race'] == 'Black') & (df['sex'] == 0),
    "Asian Male": (df['race'] == 'Asian') & (df['sex'] == 1),
    "Asian Female": (df['race'] == 'Asian') & (df['sex'] == 0),
    "ADI BIN 1": (df['deprivation_bin'] == 1),
    "ADI BIN 2": (df['deprivation_bin'] == 2)}

# Initialize result storage
auc_results = {'Model': ['PREVENT', 'XGBoost', 'Neural Network']}

# Calculate AUCs
for name, condition in subgroup_conditions.items():
    true = df.loc[condition, 'cvd']
    prevent_pred = df.loc[condition, 'prevent_cvd']
    xgb_pred = df.loc[condition, 'xgb_cvd']
    nn_pred = df.loc[condition, 'nn_cvd']

    auc_prevent = roc_auc_score(true, prevent_pred)
    auc_xgb = roc_auc_score(true, xgb_pred)
    auc_nn = roc_auc_score(true, nn_pred)

    auc_results[name] = [auc_prevent, auc_xgb, auc_nn]

# Convert to DataFrame and save
auc_df = pd.DataFrame(auc_results)

# New row of data for PREVENT(Duke)
new_row = {
    'Model': 'PREVENT(Duke)',
    'Male': 0.77252,
    'Female': 0.775511,
    'White Male': 0.76649,
    'White Female': 0.771186,
    'Black Male': 0.759498,
    'Black Female': 0.761278,
    'Asian Male': 0.774688,
    'Asian Female': 0.77488,
    'ADI BIN 1': 0.787687,
    'ADI BIN 2': 0.774764
}

# Append the new row to the existing dataframe
auc_df = pd.concat([auc_df, pd.DataFrame([new_row])], ignore_index=True)

# save
auc_df.to_csv("model_auc.csv", index=False)

auc_df

## AUC Radar Plot

In [ ]:
import matplotlib.pyplot as plt
from math import pi

# load df
auc_df = pd.read_csv("model_auc.csv")

# Define categories (exclude the first column 'Model')
categories = auc_df.columns[1:]
num_vars = len(categories)

# Compute angles for the radar chart
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # close the loop

# Compute global min and max
global_min = 0.5  # You can adjust this based on your data
global_max = 0.8  # You can adjust this based on your data
seg_count = 5
ring_labels = np.round(np.linspace(global_min, global_max, seg_count + 1), 3)

# Colors for each model
colors = ["#FFD92F", '#66C2A5', '#FC8D62', '#8DA0CB']

# Start plotting
plt.figure(figsize=(12, 10))
ax = plt.subplot(111, polar=True)

# Set category labels
plt.xticks(angles[:-1], categories, color='grey', size=10)

# Set radial labels
ax.set_rlabel_position(30)
plt.yticks(ring_labels[1:-1], [str(x) for x in ring_labels[1:-1]], color="grey", size=9)
plt.ylim(global_min, global_max)

# Plot each model
for i, row in auc_df.iterrows():
    model_name = row['Model']
    values = row[1:].values.tolist()  # Skip the 'Model' column
    values += values[:1]  # close the loop
    
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_name, color=colors[i % len(colors)])
    ax.fill(angles, values, alpha=0.25, color=colors[i % len(colors)])

# Add grid lines
ax.grid(True, linestyle='--', alpha=0.7)

# Add legend and title
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), fontsize='medium', ncol=3)
plt.title("CVD Risk Model AUC across Demographic Subgroups", size=15, y=1.05)

# Adjust layout
plt.tight_layout()
plt.show()